# Example: Control of a single spin

First, we make the necessary imports.

In [ ]:
import matplotlib.pyplot as plt

import numpy as np

from cthree.optimisation_map import OptimisationMap
from cthree.quantity import Quantity
from cthree.measurement.state_transfer_fidelity import StateTransferFidelity
from cthree.model.drive_operator import DriveOperator
from cthree.model.qubit import Qubit
from cthree.propagation.scipy_expm import ScipyExpm
from cthree.optimisers.scipy_optimiser import ScipyOptimiser

from cthree.model.closed_system import ClosedSystem

from cthree.signal.iq_mixer import IQMixer
from cthree.signal.envelopes import ConstantEnvelope

For signal generation, we define a simple cosine shaped tone generator $A \cos(\omega t)$

In [ ]:
tone = ConstantEnvelope()
gen = IQMixer(envelopes=[tone])

We can inspect the pre-defined parameters with

In [ ]:
params = gen.get_parameters()
params

in this case amplitude $A$ and frequency $\omega$.

Next, we setup the qubit system we want to control. We set the qubit frequency $\omega_q$ to be 4.8 GHz and define the Hamiltonian as $$H(t)=H_\text{drift}+H_c(t)= \frac{\omega_q}{2} \sigma_z + \Omega(t)\sigma_x$$, where $\Omega(t)$ will be supplied by the generator.

In [ ]:
FREQ = 4.8e9 * 2 * np.pi

drive = DriveOperator(gen, isLongitudinal=False)
controlled_qubit = Qubit(frequency=Quantity(FREQ, 0.8 * FREQ, 1.2 * FREQ), drives=[drive])
model = ClosedSystem(controlled_qubit)

Textbook values for implementing an $X$ rotation on this system at a time $T$ would be $\omega=\omega_q$ and $A=\pi/T$. We use some offset from these values as initial guess to demonstrate the optimization procedure.

In [ ]:
t_final = 10e-9
params[0].set_value(0.8 * np.pi / t_final)
params[2].set_value(1.01 * FREQ)

We select a propagation method, piecewise constant exponentation, and configure a state transfer problem from $\ket{0}$ to $\ket{1}$.

In [ ]:
prop = ScipyExpm(model, res=100e9)

init = np.array([[1.0], [0]])  # |0>
target = np.array([[0.0], [1]])  # |1>
zeroone = StateTransferFidelity(
    propagation=prop,
    initial_state=init,
    target_state=target,
    times=np.array([0.0, t_final]),
)

In [ ]:
def make_plot():
    """Plot the signal."""
    ts = np.linspace(0, t_final, 1001)
    states = np.reshape(prop.propagate(ts), (-1, 2))
    sig = gen.generate_signal(ts)

    fig, ax = plt.subplots(2, figsize=(4, 4), sharex=True)
    ax[0].plot(ts / 1e-9, sig)
    ax[0].set_ylabel("Field [MHz]")
    ax[1].plot(ts / 1e-9, np.abs(states) ** 2)
    ax[1].set_ylabel("Population")
    ax[-1].set_xlabel("Time [ns]")
    return fig, ax


make_plot()

As expected, we get a partial transfer and a low fidelity.

In [ ]:
zeroone.measure()

We define an optimizer and link our fidelity measure as a goal function and the parameters of the cosine tone.

In [ ]:
optmap = OptimisationMap()
optmap.add(tone, params)
opt = ScipyOptimiser(zeroone, optimisables=optmap)

In [ ]:
opt.optimise()

In [ ]:
make_plot()

We can see from the plot and optimizer output that we have found good controls.

In [ ]:
zeroone.measure()

In [ ]:
params

and parameters close to the textbook values:

In [ ]:
np.pi / t_final, FREQ